In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy
from requests import HTTPError
from loguru import logger

# 1.声明状态
class EmptyState(TypedDict):
    pass

# 2.声明结点
def node_a(state: EmptyState) -> EmptyState:
    logger.info("node_a正在运行")
    raise HTTPError

# 3.构建图
builder = StateGraph(state_schema = EmptyState)
builder.add_node("node_a",
                 node_a,
                 retry_policy = RetryPolicy(
                     max_attempts = 3,
                     jitter = False
                ))
builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

graph = builder.compile()

try:
    graph.invoke({})
except HTTPError as e:
    logger.info("重试次数耗尽:{}", e)

2026-10-25 12:26:57.156 | INFO     | __main__:node_a:13 - node_a正在运行
2026-10-25 12:26:57.657 | INFO     | __main__:node_a:13 - node_a正在运行
2026-10-25 12:26:58.659 | INFO     | __main__:node_a:13 - node_a正在运行
2026-10-25 12:26:58.661 | INFO     | __main__:<module>:32 - 重试次数耗尽:
